# 02 — Preprocessing & Feature Engineering
Demonstrates text cleaning, vectorisation, and the document-term matrix.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

from src.data.make_dataset import load_processed
from src.features.preprocess import clean_text, TextPreprocessor
from src.features.vectorize import (
    build_vectorizer, fit_transform_corpus,
    save_vectorizer, get_feature_names
)
from src.utils import load_config

cfg = load_config()
df  = load_processed()
print(f'{len(df):,} documents loaded')

## 1. Single-document cleaning walk-through

In [ ]:
raw_sample = df['raw_text'].iloc[0]
clean_sample = clean_text(raw_sample)

print('RAW (first 400 chars):')
print(raw_sample[:400])
print('\nCLEAN (first 400 chars):')
print(clean_sample[:400])

## 2. sklearn-compatible TextPreprocessor

In [ ]:
prep = TextPreprocessor(lemmatize=True)
sample_docs = df['raw_text'].head(5).tolist()
result = prep.transform(sample_docs)
for i, doc in enumerate(result):
    print(f'Doc {i}: {doc[:120]} ...')

## 3. TF-IDF Vectorisation

In [ ]:
corpus = df['clean_text'].tolist()

tfidf_vec = build_vectorizer(
    kind='tfidf',
    max_features=cfg['max_features'],
    max_df=cfg['max_df'],
    min_df=cfg['min_df'],
)
tfidf_vec, dtm_tfidf = fit_transform_corpus(tfidf_vec, corpus)
print('TF-IDF DTM shape:', dtm_tfidf.shape)
print('Sparsity: {:.2%}'.format(1 - dtm_tfidf.nnz / (dtm_tfidf.shape[0] * dtm_tfidf.shape[1])))

## 4. Count Vectorisation (for LDA)

In [ ]:
count_vec = build_vectorizer(
    kind='count',
    max_features=cfg['max_features'],
    max_df=cfg['max_df'],
    min_df=cfg['min_df'],
)
count_vec, dtm_count = fit_transform_corpus(count_vec, corpus)
print('Count DTM shape:', dtm_count.shape)

## 5. Vocabulary Statistics

In [ ]:
feature_names = get_feature_names(tfidf_vec)
print(f'Vocabulary size: {len(feature_names):,}')
print('Sample features:', feature_names[:20])

# Plot document frequency distribution
import scipy.sparse as sp
doc_freq = np.asarray((dtm_tfidf > 0).sum(axis=0)).flatten()

fig, ax = plt.subplots(figsize=(9,4))
ax.hist(doc_freq, bins=80, color='#2E86AB', edgecolor='white', alpha=0.85)
ax.set_xlabel('Document frequency')
ax.set_ylabel('Number of terms')
ax.set_title('Term Document-Frequency Distribution')
ax.set_yscale('log')
fig.tight_layout()
fig.savefig('../reports/figures/term_doc_freq.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Save the vectoriser for downstream use

In [ ]:
save_vectorizer(tfidf_vec, 'vectorizer.joblib')
print('Vectorizer saved.')